[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xuyeliu/Summer-School-Tutorial-Lab/blob/main/privacy_lab_student.ipynb)

**Running on Google Colab?** Pick a GPU runtime first (`Runtime > Change runtime type > T4 GPU`), then run the setup cell below.

- A GPU is recommended. TPU is *not* supported: Opacus computes PyTorch per-sample gradients, which do not run on TPU/XLA. CPU works too, only the from-scratch DP-SGD loop in Phase 3a is slower.
- The setup cell downloads the pre-trained checkpoints that Phase 4 needs. It does nothing when the notebook runs locally.

In [ ]:
import os
import sys

if 'google.colab' in sys.modules:
    REPO_DIR = '/content/Summer-School-Tutorial-Lab'
    if not os.path.isdir(REPO_DIR):
        !git clone -q https://github.com/xuyeliu/Summer-School-Tutorial-Lab.git {REPO_DIR}
    os.chdir(REPO_DIR)
    print('Working directory:', os.getcwd())
    print('Checkpoints available:', len([f for f in os.listdir('checkpoints') if f.endswith('.pt')]))
    !nvidia-smi -L || echo 'No GPU detected -- Runtime > Change runtime type > T4 GPU'
else:
    print('Not running on Colab, nothing to set up.')

# Privacy in Machine Learning: Hands-On Lab
## Attacking a Model with Membership Inference Attack & Defending it with Differential Privacy

**Instructions**: This notebook contains `# TODO` sections where you need to fill in code.
Each TODO requires only 1-3 lines of code. Validation cells will check your work.

**Time**: ~75 minutes total


## Lab Roadmap — read this before you start

**What you will learn:** how a trained model leaks information about the individual examples it was trained on, and how differentially private training suppresses that leak.

The lab runs in four phases, and each one uses the model produced by the previous one.

| Phase | What you do | What you should take away |
|---|---|---|
| **Phase 1** | Train a non-private baseline classifier on the member split | The model fits data it has seen better than data it has not |
| **Phase 2** | Mount a membership inference attack on that model | Members and held-out non-members get different per-sample losses | 
| **Phase 3** | Defend with DP-SGD, first by hand, then with Opacus | Clipping bounds each example's influence; noise provides the formal guarantee |
| **Phase 4** | Compare test accuracy under different epsilon values | Smaller epsilon means stronger privacy, and that can lower accuracy |

**How the session runs.** You work through one phase on your own, roughly 15 to 20 minutes each, reading the instructions and code as you go. We then stop at the checkpoint, check who finished, hand out hints or the solution to anyone stuck, and spend about three minutes on one discussion question before moving on. Ask for help whenever you want it; the goal is to understand the mechanism, not to finish first.

**Where this is going.** In this lab you learn what membership signals look like and how DP can suppress them. In the follow-up challenge you will switch to the attacker's side and try to exploit membership signals in a large language model. In the phase 4, we will proide the model to continue the exercises.


1: Set up the environment

- **Goal:** Prepares the notebook dependencies, compute device, and reproducible random state.

- **What runs:** Installs the required packages; imports PyTorch, MedMNIST, Opacus, scikit-learn, NumPy, and Matplotlib; selects CUDA, MPS, or CPU; and sets random seeds.

- **Outcome:** The environment reports the selected device and confirms that all packages were imported successfully.

- **TODO:** N/A


In [ ]:
%pip install torch torchvision medmnist opacus scikit-learn matplotlib numpy --quiet

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve
from opacus import PrivacyEngine          # handles DP-SGD for us in Phase 3
import medmnist
from medmnist import BreastMNIST
import warnings
warnings.filterwarnings('ignore')

# Prefer GPU (CUDA), then Apple Silicon (MPS), then CPU as a fallback.
if torch.cuda.is_available():
    device = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f"Using device: {device}")

# Fix seeds so every student gets reproducible, comparable numbers.
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print("All packages imported successfully!")


## Phase 0: Warm-up and Orientation (~5 min)
**Phase 0 Goal**: Confirm environment works, explore the dataset

2: Load and inspect BreastMNIST

- **Goal:** Loads the official BreastMNIST splits, normalizes the images, and examines class balance.

- **What runs:** Uses `transforms.Compose`, `BreastMNIST`, `np.unique`, and Matplotlib to prepare the data and plot Malignant versus Normal/Benign counts.

- **Outcome:** The dataset sizes, tensor shapes, and training-class distribution are displayed.

- **TODO:** N/A

**About BreastMNIST:**

- BreastMNIST is based on 780 breast ultrasound images. 

- The original labels are normal, benign, and malignant. At 28×28 resolution the task is simplified to binary classification: malignant (0) versus normal/benign (1).

- Source images are 1×500×500 grayscale ultrasounds, resized to 1×28×28. The official split is 546 / 78 / 156 (train / val / test).

- label information: {'0': 'malignant', '1': 'normal, benign'}



In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

train_dataset = BreastMNIST(split='train', transform=transform, download=True)
val_official = BreastMNIST(split='val', transform=transform, download=True)
test_dataset = BreastMNIST(split='test', transform=transform, download=True)

print(f"Training set size: {len(train_dataset)}")
print(f"Validation set size: {len(val_official)}")
print(f"Test set size: {len(test_dataset)}")
print(f"Image shape: {train_dataset[0][0].shape}")
print(f"Label shape: {train_dataset[0][1].shape}")

# Class imbalance matters: a skewed dataset can inflate accuracy,
# so we visualize how many Malignant vs Normal/Benign examples we have.
train_labels = np.array([train_dataset[i][1].item() for i in range(len(train_dataset))])
unique, counts = np.unique(train_labels, return_counts=True)

fig, ax = plt.subplots(figsize=(6, 4))
class_names = ['Malignant', 'Normal/Benign']
ax.bar(class_names, counts, color=['steelblue', 'coral'])
ax.set_ylabel('Count')
ax.set_title('Training Set Class Distribution')
for i, (name, count) in enumerate(zip(class_names, counts)):
    ax.text(i, count + 5, str(count), ha='center', fontsize=12)
plt.tight_layout()
plt.show()


3: Preview sample ultrasounds

- **Goal:** Provides a visual sanity check of the normalized BreastMNIST images and labels.

- **What runs:** Uses `plt.subplots`, iterates through the first eight training samples, reverses normalization for display, and renders each image with `imshow`.

- **Outcome:** A 2-by-4 gallery of labeled Malignant and Normal/Benign ultrasounds is shown.

- **TODO:** N/A


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i, ax in enumerate(axes.flat):
    img, label = train_dataset[i]
    # Undo the Normalize step so the ultrasound displays with natural contrast.
    img_display = img.squeeze().numpy() * 0.5 + 0.5
    ax.imshow(img_display, cmap='gray')
    ax.set_title(f"{'Normal/Benign' if label.item() == 1 else 'Malignant'}")
    ax.axis('off')
plt.suptitle('Sample Images from BreastMNIST', fontsize=14)
plt.tight_layout()
plt.show()


4: Create membership-inference splits

- **Goal:** Divides the official BreastMNIST **train split** into disjoint member and non-member halves for the privacy experiment, and points utility evaluation at the official test set.

- **What runs:** Uses only the 546-image official train splits as **50/50** into members and non-members. The official BreastMNIST **test** split is used for test accuracy.

- **Outcome:** Member / non-member / test loaders are created, and the size of each split is printed.

- **TODO:** N/A

### This split is the foundation of the whole lab

**The official train split** is divided into two equal groups:

- **`member_dataset` (50%)** — the *only* data the target model is ever trained on. These samples are the **members**.
- **`nonmember_dataset` (50%)** — held out completely. The model never sees these samples. They are the **non-members**.

**Test accuracy** uses the official BreastMNIST **test** set (never part of the membership pool). The official **val** split is not part of the membership pool or the utility evaluation.

Members and non-members are drawn from the same distribution, so they are statistically indistinguishable before training. The *only* difference between the two groups is **whether the model was trained on them**.

That is exactly what the attacker in Phase 2 will try to detect:

> Given an example, decide whether it was used to train this model.


In [ ]:
# Membership pool = official train split (546 images); 50/50 members/non-members with seed 500.
# Utility (test accuracy) uses the official BreastMNIST test split.
pool = train_dataset
total_size = len(pool)
mid = total_size // 2

indices = np.arange(total_size)
np.random.seed(500)
np.random.shuffle(indices)
np.random.seed(None)

member_indices = indices[:mid]
nonmember_indices = indices[mid:]

member_dataset = Subset(pool, member_indices)        # MEMBERS: used for training
nonmember_dataset = Subset(pool, nonmember_indices)  # NON-MEMBERS: never trained on

# member_loader is the ONLY loader we will ever pass to an optimizer.
# shuffle=True only for the set we train on; evaluation loaders stay ordered.
member_loader = DataLoader(member_dataset, batch_size=64, shuffle=True)
nonmember_loader = DataLoader(nonmember_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"Membership pool (train split only):                  {len(pool)}")
print(f"Members     -> TRAINED ON     (attacker's label 1): {len(member_dataset)}")
print(f"Non-members -> HELD OUT       (attacker's label 0): {len(nonmember_dataset)}")
print(f"Official test set (utility / generalization):      {len(test_dataset)}")
print()
print("The attacker's question: given an example, was it one of the members?")


5: Validate the data setup

- **Goal:** Checks that Phase 0 produced usable member and non-member datasets and selected a compute device.

- **What runs:** Runs three conditional checks with `len(...)` and the `device` value, then counts and prints passing checks.

- **Outcome:** A Phase 0 validation summary reports up to three passes and confirms completion when all checks succeed.

- **TODO:** N/A


In [ ]:
print("=" * 50)
print("PHASE 0 VALIDATION")
print("=" * 50)
checks = 0
if len(member_dataset) > 0:
    print(f"[PASS] Member dataset loaded: {len(member_dataset)} samples")
    checks += 1
if len(nonmember_dataset) > 0:
    print(f"[PASS] Non-member dataset loaded: {len(nonmember_dataset)} samples")
    checks += 1
if device is not None:
    print(f"[PASS] Device set: {device}")
    checks += 1
print(f"\nResult: {checks}/3 checks passed.")
if checks == 3:
    print("Phase 0 COMPLETE!")
print("=" * 50)

6: Define the baseline model

- **Goal:** Defines the fully connected neural network used for binary BreastMNIST classification.

- **What runs:** Creates the `SimpleFC` class and `create_model()` helper, moves a test model to the selected device, and counts its parameters.

- **Outcome:** The model architecture becomes available for later training, and its total parameter count is printed.

- **TODO:** N/A


In [ ]:
class SimpleFC(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28*28, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 1)   # single logit -> malignant vs normal/benign
    def forward(self, x):
        x = x.view(x.size(0), -1)      # flatten the 28x28 image into a vector
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

def create_model():
    return SimpleFC().to(device)

test_model = create_model()
total_params = sum(p.numel() for p in test_model.parameters())
print(f"Total parameters: {total_params:,}")
del test_model


## Phase 1: Train the Non-Private Baseline (~15 min)

**Phase 1 goal:** Train the non-private baseline model and examine its train/test behavior.

We will use this trained model in the next phase to study membership inference. Phase 1 itself is only about training a classifier and looking at how it performs on data it has seen versus data it has not.

**What to watch for:** whether accuracy on the member (training) split ends up higher than accuracy on the test set, and by how much.

**TODO in this phase:** TODO 1, the training step inside `train_one_epoch`.


7: Define training and evaluation helpers

- **Goal:** Introduces reusable functions for one training epoch and for classification-accuracy evaluation.

- **What runs:** `train_one_epoch` performs optimization over a data loader, while `evaluate_accuracy` uses `torch.no_grad`, sigmoid thresholding, and label comparisons.

- **Outcome:** Once completed, the helpers will return average training loss and percentage accuracy for later phases.

- **TODO:** Complete TODO 1 by adding the forward pass, loss calculation, backpropagation, and optimizer step in `train_one_epoch`.


In [ ]:
def train_one_epoch(model, dataloader, optimizer, criterion):
    """Train for one epoch, return average loss."""
    model.train()
    total_loss = 0.0
    n_batches = 0
    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.float().to(device).view(-1, 1)   # shape (B,1) for BCE

        # TODO 1: Complete the training step (3 lines)
        # Hint: zero gradients, compute loss from model outputs, backprop and step
        optimizer.zero_grad()
        outputs = None  # <-- Replace None with forward pass
        loss = None     # <-- Replace None with loss computation
        # <-- Add backward and step calls

        total_loss += loss.item()
        n_batches += 1
    return total_loss / n_batches

def evaluate_accuracy(model, dataloader):
    """Compute accuracy as percentage."""
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():   # no gradients needed at eval -> faster, less memory
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.float().to(device).view(-1, 1)
            outputs = model(images)
            # logit >= 0  <=>  sigmoid(logit) >= 0.5  -> predict class 1.
            preds = (torch.sigmoid(outputs) >= 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return 100.0 * correct / total

print("Training functions defined.")

8: Train the non-private baseline

- **Goal:** Trains the baseline model on the **member split only**, then compares its accuracy on that seen data against unseen test data.

- **What runs:** Creates a model with `create_model`, optimizes it with Adam and `BCEWithLogitsLoss`, calls `train_one_epoch` and `evaluate_accuracy` for 60 epochs, and plots both accuracy curves.

- **Outcome:** Training progress, final train/test accuracy, and a visualization of the difference between them.

- **TODO:** N/A


In [ ]:
model = create_model()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()   # combines sigmoid + binary cross-entropy

NUM_EPOCHS = 60
train_accs = []
test_accs = []

print(f"Training baseline model for {NUM_EPOCHS} epochs on the MEMBER split only...")
for epoch in range(NUM_EPOCHS):
    # member_loader = the members. The non-member split never reaches the optimizer,
    # which is what makes it a valid comparison group for the Phase 2 attack.
    loss = train_one_epoch(model, member_loader, optimizer, criterion)
    train_acc = evaluate_accuracy(model, member_loader)   # accuracy on MEMBERS (seen)
    test_acc = evaluate_accuracy(model, test_loader)      # accuracy on unseen data
    train_accs.append(train_acc)
    test_accs.append(test_acc)
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{NUM_EPOCHS}: Loss={loss:.4f}, Train={train_acc:.1f}%, Test={test_acc:.1f}%")

# Plot the two curves; the shaded band highlights the train-test gap.
plt.figure(figsize=(8, 5))
plt.plot(range(1, NUM_EPOCHS+1), train_accs, 'b-o', label='Train (members, seen)', markersize=4)
plt.plot(range(1, NUM_EPOCHS+1), test_accs, 'r-o', label='Test (unseen)', markersize=4)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Accuracy (%)', fontsize=12)
plt.title('Baseline Model: Members (seen) vs Test (unseen)', fontsize=14)
plt.grid(True, alpha=0.3)
gap_final = train_accs[-1] - test_accs[-1]
plt.axhspan(test_accs[-1], train_accs[-1], alpha=0.1, color='red',
            label=f'Train-test gap = {gap_final:.1f}%')
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f"\nFinal: Train={train_accs[-1]:.1f}%, Test={test_accs[-1]:.1f}%, Gap={gap_final:.1f}%")


9: Check the baseline training result

- **Goal:** Confirms that the baseline actually fit its training data and that train accuracy ended up above test accuracy.

- **What runs:** Reads the final accuracy values, prints both, and applies two soft checks.

- **Outcome:** A Phase 1 summary reporting whether the baseline behaves as the next phase expects.

- **TODO:** N/A


In [ ]:
print("=" * 50)
print("PHASE 1 VALIDATION")
print("=" * 50)
# These are SOFT checks on the qualitative pattern, not on exact numbers. The seed fixes
# initialization and shuffling, but results still move by a few points across hardware
# (CPU vs GPU, different GPU models, cuDNN kernels) and library versions. Do not worry
# if your accuracies differ from a neighbour's by a few points.
checks_in_range = 0
total_checks = 2

final_train_acc = train_accs[-1]
final_test_acc = test_accs[-1]
gap = final_train_acc - final_test_acc

print(f"Train (member) accuracy: {final_train_acc:.1f}%")
print(f"Test accuracy:           {final_test_acc:.1f}%")
print(f"Train-test gap:          {gap:.1f} points")
print("-" * 50)

if final_train_acc > 90:
    print(f"[PASS] The model fits its training data (train accuracy > 90%)")
    checks_in_range += 1
else:
    print(f"[WARN] Train accuracy is low, so the model barely fit the member data.")
    print("       Re-check TODO 1, or train for more epochs.")

if gap > 2:
    print(f"[PASS] Train accuracy exceeds test accuracy by {gap:.1f} points")
    checks_in_range += 1
else:
    print(f"[WARN] The train-test gap is small ({gap:.1f} points).")
    print("       A small gap makes the Phase 2 attack signal weak; more epochs widen it.")

print("-" * 50)
print(f"Result: {checks_in_range}/{total_checks} checks passed.")
if checks_in_range == total_checks:
    print("Phase 1 COMPLETE!")
else:
    print("Phase 1 can still continue: warnings flag numbers to interpret, not code to fix.")
print("=" * 50)


## Phase 2: Mounting a Membership Inference Attack (~15 min)

**Recall from Cell 4:** the training pool was split into a **member** group and a **non-member** group. The baseline model you just trained saw *only* the members. The non-members were held out and never reached the optimizer.

The attacker's task:

> Given an example, decide whether it was used to train this model.

**Phase 2 goal:** find a quantity that behaves differently on members than on non-members, and turn it into an attack signal.

**The signal we use: per-sample loss.** The model was optimized to fit the members, so it is usually more confident on them. That means **member loss tends to be lower than non-member loss**. We summarize the effect with the loss ratio:

`loss ratio = mean(loss on non-members) / mean(loss on members)`

A ratio near 1 means the two groups look alike, so there is little to exploit. A ratio well above 1 means the model treats its training data measurably differently.

**Why the risk is real:** confirming that a record was in a training set can reveal that a person was, for example, a patient in a breast-cancer screening study, even when the training data itself is never released.

Cell 11 also prints an AUC. Hold that number for the checkpoint. Do not treat it as the main result yet.

**TODOs in this phase:** TODO 2 and TODO 3.


10: Define the membership signal

- **Goal:** Defines per-sample loss as the confidence signal used by the membership-inference attack.

- **What runs:** `get_membership_signal` runs the model in evaluation mode and is intended to call `F.binary_cross_entropy_with_logits` with `reduction='none'`.

- **Outcome:** The completed function will return one loss value per sample, where lower loss suggests membership.

- **TODO:** Complete TODO 2 by computing the unreduced binary cross-entropy loss for every sample.


In [ ]:
def get_membership_signal(model, dataloader):
    """Compute per-sample loss. Lower loss = more confident = more likely a member."""
    signals = []
    model.eval()
    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.float().to(device).view(-1, 1)
            outputs = model(images)

            # TODO 2: Compute per-sample loss (1 line)
            # Hint: Use F.binary_cross_entropy_with_logits with reduction='none'
            # (reduction='none' keeps ONE loss per sample instead of averaging)
            per_sample_loss = None  # <-- Replace None

            signals.extend(per_sample_loss.cpu().numpy().flatten())
    return np.array(signals)

print("get_membership_signal function defined.")

11: Score the baseline attack

- **Goal:** Applies the membership signal to member and non-member data, then builds the ROC curve one threshold at a time so every step of the attack is visible.

- **What runs:** Calls `get_membership_signal`, constructs labels and negated-loss scores, loops over candidate thresholds to count true and false positives, computes TPR/FPR and the ROC area by hand, and marks the threshold maximizing `TPR - FPR`.

- **Outcome:** The cell reports the loss ratio and attack AUC, plots the ROC curve, and highlights the selected loss threshold `t`.

- **TODO:** Complete TODO 3 by constructing labels and scores, filling in the threshold-loop counts, and calculating the non-member-to-member loss ratio.


In [ ]:
member_signals = get_membership_signal(model, member_loader)
nonmember_signals = get_membership_signal(model, nonmember_loader)

# TODO 3a: Build the labels, scores, and loss ratio.
# Members have label 1; non-members have label 0.
# Negate loss because a higher score should mean "more likely a member".
attack_labels = None   # <-- concatenate 1s for members and 0s for non-members
attack_scores = None   # <-- concatenate the negated member/non-member losses
loss_ratio = None      # <-- mean(nonmember loss) / mean(member loss)

# Try every distinct attack score as a decision threshold. The two infinite
# endpoints make the ROC curve start at (0, 0) and finish at (1, 1).
attack_thresholds = np.concatenate((
    [np.inf],
    np.sort(np.unique(attack_scores))[::-1],
    [-np.inf],
))

tp_list = []
fp_list = []
tpr_list = []
fpr_list = []
num_members = np.sum(attack_labels == 1)
num_nonmembers = np.sum(attack_labels == 0)

for threshold in attack_thresholds:
    predicted_member = attack_scores >= threshold

    # TODO 3b: Count true positives and false positives at this threshold.
    tp = None   # <-- sum predicted members whose true label is 1
    fp = None   # <-- sum predicted members whose true label is 0

    tp_list.append(int(tp))
    fp_list.append(int(fp))
    tpr_list.append(tp / num_members)
    fpr_list.append(fp / num_nonmembers)

tpr_array = np.array(tpr_list)
fpr_array = np.array(fpr_list)
threshold_list = tpr_array - fpr_array
best_threshold_index = int(np.argmax(threshold_list))
best_score_threshold = float(attack_thresholds[best_threshold_index])
selected_loss_threshold = -best_score_threshold

# TODO 3c: Add the trapezoid areas under the ROC curve without roc_auc_score().
auc_area = 0.0
for i in range(1, len(fpr_list)):
    width = None           # <-- change in FPR from the previous point
    average_height = None  # <-- average of the two neighboring TPR values
    auc_area += width * average_height

print("MIA Attack Results")
print("-" * 52)
print(f"  Mean loss - members (seen):       {np.mean(member_signals):.4f}")
print(f"  Mean loss - non-members (unseen): {np.mean(nonmember_signals):.4f}")
print(f"  Loss ratio: {loss_ratio:.2f}x  (1.0 = no difference, higher = more leakage)")
print(f"  AUC:        {auc_area:.4f}  (0.5 = random guess)")
print(f"  Chosen t:   {selected_loss_threshold:.6f}  (maximizes TPR - FPR)")

plt.figure(figsize=(6.5, 5.5))
plt.plot(fpr_array, tpr_array, linewidth=2, label=f'ROC curve (AUC = {auc_area:.3f})')
plt.plot([0, 1], [0, 1], 'k--', alpha=0.6, label='Random guess')
plt.scatter(
    fpr_array[best_threshold_index],
    tpr_array[best_threshold_index],
    color='red',
    s=70,
    zorder=3,
    label=f'Chosen t = {selected_loss_threshold:.4g}',
)
plt.annotate(
    f'max(TPR - FPR) = {threshold_list[best_threshold_index]:.3f}',
    (fpr_array[best_threshold_index], tpr_array[best_threshold_index]),
    xytext=(12, -28),
    textcoords='offset points',
    color='red',
)
plt.xlabel('False positive rate (FPR)')
plt.ylabel('True positive rate (TPR)')
plt.title('Membership inference attack ROC curve')
plt.xlim(0, 1)
plt.ylim(0, 1.02)
plt.grid(True, alpha=0.3)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

# Keep mia_auc at the end: later cells reuse this baseline value.
mia_auc = auc_area

12: Turn the ROC curve into a threshold attack

- **Goal:** Converts the Phase 2 ranking score into an actual yes/no membership attack using the threshold selected in the preceding ROC plot.

- **What runs:** Reuses the manually computed `TPR`, `FPR`, and `argmax(TPR - FPR)` result, classifies every sample as a member when `loss <= t`, and counts correct predictions.

- **Outcome:** Prints `t`, attack accuracy, and the correct-count, then shows only the loss histogram and its decision threshold.

- **TODO:** N/A

### Attack rule

1. Score each example with `s = -loss` (higher ⇒ more member-like).
2. Choose the point maximizing `TPR - FPR` on the ROC curve.
3. Convert its score threshold back to a loss threshold `t` and predict **MEMBER** if `loss <= t`.

Random guessing on a balanced member/non-member set is 50%. If accuracy is clearly above that, the AUC yields a working attack.

**How to read the histogram.** The y-axis is **count** (number of samples per bin), not density. Many members sit at near-zero loss after overfitting, so the leftmost bin is tall. The dashed line is the selected decision threshold: everything to its left is called MEMBER.


In [ ]:
from sklearn.metrics import confusion_matrix

# Reuse the operating point selected explicitly in the preceding ROC loop.
k_thr = best_threshold_index
t = selected_loss_threshold

all_loss = np.concatenate([member_signals, nonmember_signals])
threshold_preds = (all_loss <= t).astype(float)
threshold_attack_correct = int((threshold_preds == attack_labels).sum())
threshold_attack_acc = threshold_attack_correct / len(attack_labels)
tn, fp, fn, tp = confusion_matrix(attack_labels, threshold_preds).ravel()

print("Threshold membership attack (baseline)")
print("-" * 52)
print(f"  AUC (from the ROC loop):         {mia_auc:.4f}")
print(f"  Chosen loss threshold t:         {t:.6f}")
print(f"  TPR @ t:                         {tpr_array[k_thr]:.3f}")
print(f"  FPR @ t:                         {fpr_array[k_thr]:.3f}")
print(f"  Correct predictions:             {threshold_attack_correct} / {len(attack_labels)}")
print(f"  Attack accuracy:                 {100*threshold_attack_acc:.1f}%")
print(f"  Random guess (balanced):         50.0%")
print()
better = threshold_attack_acc > 0.5
print(f"Is this attack better than random guess?  {'YES' if better else 'NO'}")
print(f"  (TP={tp}, FN={fn}, TN={tn}, FP={fp})")

# Show only the loss distributions and the selected threshold.
fig, ax = plt.subplots(figsize=(7, 4.5))

LOSS_FLOOR = 1e-8
m_plot = np.maximum(member_signals, LOSS_FLOOR)
nm_plot = np.maximum(nonmember_signals, LOSS_FLOOR)
# Equal-width bins in log10(loss) space give a fair comparison of counts.
log_lo = np.log10(LOSS_FLOOR)
log_hi = np.log10(max(m_plot.max(), nm_plot.max(), t) * 1.5)
bins = np.logspace(log_lo, log_hi, 35)

ax.hist(m_plot, bins=bins, alpha=0.55, label=f'Members (n={len(m_plot)})', color='C0')
ax.hist(nm_plot, bins=bins, alpha=0.55, label=f'Non-members (n={len(nm_plot)})', color='C1')
ax.axvline(t, color='k', ls='--', lw=2, label=f't = {t:.4f}')
ax.set_xscale('log')
ax.set_xlabel('Per-sample loss (log scale)')
ax.set_ylabel('Count')
ax.set_title('Loss distributions and threshold t')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

13: Validate the baseline attack

- **Goal:** Checks that the computed membership signals exhibit the expected privacy-leakage pattern.

- **What runs:** Compares average losses, tests whether the loss ratio exceeds 1.3, verifies both signal arrays are nonempty, and totals the passing checks.

- **Outcome:** The Phase 2 summary reports whether the membership-inference attack works as expected.

- **TODO:** N/A


In [ ]:
print("=" * 50)
print("PHASE 2 VALIDATION")
print("=" * 50)
checks_passed_2 = 0
total_checks_2 = 3

if np.mean(member_signals) < np.mean(nonmember_signals):
    print(f"[PASS] Members have lower avg loss ({np.mean(member_signals):.4f} < {np.mean(nonmember_signals):.4f})")
    checks_passed_2 += 1
else:
    print(f"[FAIL] Expected members to have lower loss than non-members")

if loss_ratio > 1.3:
    print(f"[PASS] Loss ratio = {loss_ratio:.2f}x (target: > 1.3x)")
    checks_passed_2 += 1
else:
    print(f"[FAIL] Loss ratio = {loss_ratio:.2f}x (target: > 1.3x)")

if len(member_signals) > 0 and len(nonmember_signals) > 0:
    print(f"[PASS] Signals computed: {len(member_signals)} members, {len(nonmember_signals)} non-members")
    checks_passed_2 += 1
else:
    print(f"[FAIL] Signals are empty")

print("-" * 50)
print(f"Result: {checks_passed_2}/{total_checks_2} checks passed.")
if checks_passed_2 == total_checks_2:
    print("Phase 2 COMPLETE!")
print("=" * 50)

### Phase 2 Checkpoint — Discussion

Stop here. Look at your **AUC** and **threshold-attack accuracy**.

1. Can an outsider tell who was in the training set — yes or no? Which number convinced you?
2. Would you release this model on real patient ultrasounds? Why?

<details>
<summary>After you discuss — click to reveal</summary>

Yes — it leaks. AUC ~0.65 and ~63% threshold accuracy both beat coin-flip (50%), so membership is guessable from loss alone. Most teams would not ship this without a defense. Next phase: DP-SGD, and we check whether the same attack gets harder.

</details>

---


## Phase 3a: Implementing DP-SGD From Scratch (~20 min)

**Phase 3a Goal:** Build an intuitive understanding of DP-SGD by implementing its privacy mechanisms directly.

**DP-SGD in one sentence:**

> DP-SGD first clips each example's gradient contribution, aggregates the clipped gradients, and then adds Gaussian noise to the aggregated gradient before the parameter update.

Two details in that sentence are the ones people most often get wrong:

- **Clipping is per example, not per batch.** Each sample's own gradient is rescaled to norm at most `C` *before* it is added to anything.
- **Noise is added once, to the aggregated gradient.** It is not drawn separately for each example, and it is applied before the optimizer updates the parameters.

Standard SGD combines the gradients from a mini-batch and uses them to update the model. DP-SGD modifies this process so that no single training example has too much influence and individual contributions are difficult to recover.

**What we will focus on:**

- **Per-sample gradients:** Compute a separate gradient for each training example instead of immediately averaging the whole batch.
- **Gradient clipping:** Bound each sample's gradient norm so that one patient cannot dominate the model update.
- **Gradient aggregation:** Sum the clipped gradients only after every sample's influence has been limited.
- **Gaussian noise:** Add calibrated random noise to the aggregated gradient to mask individual contributions.
- **Private model updates:** Average the noisy gradient and use it to update the model parameters.

Implementing these steps from scratch makes it easier to see where the privacy protection comes from and how clipping and noise change ordinary training.

**Coming next:** In Phase 3b, we will use the Opacus library to automate per-sample clipping, noise addition, and privacy accounting. We will then compare the manual and library-based approaches.

> **Note:** This educational implementation uses an explicit per-sample Python loop, so it may run slowly. It is designed to clarify the mechanism rather than optimize performance.


**Understanding Check**

DP-SGD protects privacy with two steps, and the order is what makes the guarantee work:

1. **Bound sensitivity:** clip each per-sample gradient so no single sample can dominate the update.
2. **Add noise:** perturb the bounded aggregate once, so individual contributions are masked.

**Think:** why must clipping happen first? (Hint: the noise scale is calibrated to the clipping bound `C`. If one sample's gradient can be arbitrarily large, no fixed amount of noise is enough to hide it.)

14: Implement DP-SGD from scratch

- **Goal:** Demonstrates the mechanics of private training by manually clipping per-sample gradients and adding Gaussian noise.

- **What runs:** Uses `get_noise_multiplier`, computes individual gradients in `compute_dp_gradients`, accumulates noisy gradients, and trains a fresh model with manual SGD updates.

- **Outcome:** A manually private model is trained and evaluated; the following cell measures its attack AUC, selects a threshold, and draws the ROC and loss histogram.

- **TODO:** Complete TODO 4a by clipping each sample's gradients, and TODO 4b by drawing Gaussian noise with the specified standard deviation.

In [ ]:
from opacus.accountants.utils import get_noise_multiplier

# --- Hyper-parameters for the manual run ---------------------------------
MANUAL_EPOCHS = 15            # per-sample loop is slow, so keep this modest
MANUAL_MAX_GRAD_NORM = 1.0    # C: the clipping bound (limits each sample's influence)
MANUAL_EPSILON = 8.0          # target privacy budget for this demo
MANUAL_DELTA = 1e-5
manual_sample_rate = 64 / len(member_dataset)  # batch_size / dataset_size

# Ask Opacus's accountant ONLY for the required noise scale (not for training).
manual_noise_multiplier = get_noise_multiplier(
    target_epsilon=MANUAL_EPSILON,
    target_delta=MANUAL_DELTA,
    sample_rate=manual_sample_rate,
    epochs=MANUAL_EPOCHS,
)
print(f"Computed noise multiplier (sigma): {manual_noise_multiplier:.4f}")


def compute_dp_gradients(model, images, labels, criterion, max_grad_norm, noise_multiplier):
    """Return the DP gradient for one mini-batch: per-sample clip, sum, add noise, average."""
    accumulated = {name: torch.zeros_like(p) for name, p in model.named_parameters()}
    batch_size = images.size(0)

    # --- Process the batch ONE sample at a time so we can clip each individually ---
    for i in range(batch_size):
        model.zero_grad()
        output = model(images[i:i + 1])          # forward on a single example
        loss = criterion(output, labels[i:i + 1])
        loss.backward()                          # gradient of THIS sample only

        # ============================================================
        # TODO 4a: CLIP this single sample's gradient so its L2 norm <= max_grad_norm.
        #          This bounds how much one patient can influence the model (sensitivity).
        #          ORDER MATTERS: clip HERE, BEFORE the gradient is added to
        #          `accumulated` in the loop below. Clipping the aggregate instead
        #          would bound the batch, not any individual sample.
        # Hint: torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        # ============================================================
        pass  # <-- Replace this line with the clipping call

        for name, param in model.named_parameters():
            accumulated[name] += param.grad      # add the clipped gradient to the sum

    # This loop is OUTSIDE the per-sample loop above: every sample has already been
    # clipped and summed, so we now perturb the aggregate once per parameter.
    for name, param in model.named_parameters():
        # ============================================================
        # TODO 4b: DRAW Gaussian noise with mean 0 and std = noise_multiplier * max_grad_norm.
        #          This masks individual contributions and gives the formal guarantee.
        #          Draw it ONCE for the aggregated gradient, NOT once per sample:
        #          per-sample noise would inject batch_size times more noise than the
        #          accountant assumed, so the reported epsilon would no longer match.
        # Hint: torch.normal(0.0, noise_multiplier * max_grad_norm,
        #                    size=param.shape, device=param.device)
        # ============================================================
        noise = None  # <-- Replace None with the noise tensor
        accumulated[name] = (accumulated[name] + noise) / batch_size
    return accumulated


# --- Train a fresh model with our hand-written DP gradients ---------------
manual_loader = DataLoader(member_dataset, batch_size=64, shuffle=True)
model_manual = create_model()                    # same architecture as every other phase
optimizer_manual = torch.optim.SGD(model_manual.parameters(), lr=0.2)
criterion_manual = nn.BCEWithLogitsLoss()

print(f"Training with manual DP-SGD for {MANUAL_EPOCHS} epochs (slower -- per-sample loop)...")
for epoch in range(MANUAL_EPOCHS):
    model_manual.train()
    for images, labels in manual_loader:
        images = images.to(device)
        labels = labels.float().to(device).view(-1, 1)
        grads = compute_dp_gradients(model_manual, images, labels, criterion_manual,
                                     MANUAL_MAX_GRAD_NORM, manual_noise_multiplier)
        for name, param in model_manual.named_parameters():
            param.grad = grads[name]             # inject our DP gradient
        optimizer_manual.step()                  # ordinary SGD update using it

manual_test_acc = evaluate_accuracy(model_manual, test_loader)
print(f"Manual DP-SGD test accuracy: {manual_test_acc:.1f}%")

# Keep the numerical comparison used by later validation cells, without the CDF plot.
member_signals_manual = get_membership_signal(model_manual, member_loader)
nonmember_signals_manual = get_membership_signal(model_manual, nonmember_loader)
loss_ratio_manual = np.mean(nonmember_signals_manual) / (np.mean(member_signals_manual) + 1e-8)

print(f"Baseline loss ratio: {loss_ratio:.2f}x")
print(f"Manual DP-SGD loss ratio: {loss_ratio_manual:.2f}x")
print(f"Manual DP-SGD test accuracy: {manual_test_acc:.1f}%")

14b: Attack the manual DP-SGD model

- **Goal:** Repeats the transparent Phase 2 membership attack on the manually trained DP model.

- **What runs:** Loops over every candidate score threshold, counts TP and FP, builds the ROC curve, computes its area with trapezoids, selects the loss threshold maximizing `TPR - FPR`, and draws side-by-side baseline/manual-DP ROC and loss-histogram comparisons.

- **Outcome:** Each comparison places the baseline on the left and manual DP-SGD on the right, using shared axes and shared histogram bins.

- **TODO:** N/A


In [ ]:
# Build the manual-DP attack inputs: higher score means more likely to be a member.
attack_labels_manual = np.concatenate([
    np.ones(len(member_signals_manual)),
    np.zeros(len(nonmember_signals_manual)),
])
attack_scores_manual = np.concatenate([-member_signals_manual, -nonmember_signals_manual])

manual_attack_thresholds = np.concatenate((
    [np.inf],
    np.sort(np.unique(attack_scores_manual))[::-1],
    [-np.inf],
))

tp_list_manual = []
fp_list_manual = []
tpr_list_manual = []
fpr_list_manual = []
num_members_manual = np.sum(attack_labels_manual == 1)
num_nonmembers_manual = np.sum(attack_labels_manual == 0)

for threshold in manual_attack_thresholds:
    predicted_member = attack_scores_manual >= threshold
    tp_manual = np.sum(predicted_member & (attack_labels_manual == 1))
    fp_manual = np.sum(predicted_member & (attack_labels_manual == 0))

    tp_list_manual.append(int(tp_manual))
    fp_list_manual.append(int(fp_manual))
    tpr_list_manual.append(tp_manual / num_members_manual)
    fpr_list_manual.append(fp_manual / num_nonmembers_manual)

tpr_array_manual = np.array(tpr_list_manual)
fpr_array_manual = np.array(fpr_list_manual)
threshold_list_manual = tpr_array_manual - fpr_array_manual
best_threshold_index_manual = int(np.argmax(threshold_list_manual))
best_score_threshold_manual = float(manual_attack_thresholds[best_threshold_index_manual])
t_manual = -best_score_threshold_manual

# Compute AUC explicitly as the sum of trapezoid areas.
mia_auc_manual = 0.0
for i in range(1, len(fpr_list_manual)):
    width = fpr_list_manual[i] - fpr_list_manual[i - 1]
    average_height = (tpr_list_manual[i] + tpr_list_manual[i - 1]) / 2
    mia_auc_manual += width * average_height

all_loss_manual = np.concatenate([member_signals_manual, nonmember_signals_manual])
threshold_preds_manual = (all_loss_manual <= t_manual).astype(float)
threshold_correct_manual = int((threshold_preds_manual == attack_labels_manual).sum())
threshold_acc_manual = threshold_correct_manual / len(attack_labels_manual)
tn_manual, fp_manual, fn_manual, tp_manual = confusion_matrix(
    attack_labels_manual, threshold_preds_manual
).ravel()

print("Threshold membership attack (manual DP-SGD)")
print("-" * 52)
print(f"  AUC:                            {mia_auc_manual:.4f}")
print(f"  Chosen loss threshold t:        {t_manual:.6f}")
print(f"  TPR @ t:                        {tpr_array_manual[best_threshold_index_manual]:.3f}")
print(f"  FPR @ t:                        {fpr_array_manual[best_threshold_index_manual]:.3f}")
print(f"  Correct predictions:            {threshold_correct_manual} / {len(attack_labels_manual)}")
print(f"  Attack accuracy:                {100 * threshold_acc_manual:.1f}%")
print(f"  (TP={tp_manual}, FN={fn_manual}, TN={tn_manual}, FP={fp_manual})")

# ROC comparison: baseline on the left, manual DP-SGD on the right.
fig, (ax_base_roc, ax_manual_roc) = plt.subplots(
    1, 2, figsize=(13, 5.5), sharex=True, sharey=True
)

ax_base_roc.plot(
    fpr_array,
    tpr_array,
    linewidth=2,
    label=f'ROC curve (AUC = {mia_auc:.3f})',
)
ax_base_roc.plot([0, 1], [0, 1], 'k--', alpha=0.6, label='Random guess')
ax_base_roc.scatter(
    fpr_array[best_threshold_index],
    tpr_array[best_threshold_index],
    color='red',
    s=70,
    zorder=3,
    label=f'Chosen t = {t:.4g}',
)
ax_base_roc.annotate(
    f'max(TPR - FPR) = {threshold_list[best_threshold_index]:.3f}',
    (fpr_array[best_threshold_index], tpr_array[best_threshold_index]),
    xytext=(10, -28),
    textcoords='offset points',
    color='red',
)
ax_base_roc.set_title('Baseline (no DP)')
ax_base_roc.set_xlabel('False positive rate (FPR)')
ax_base_roc.set_ylabel('True positive rate (TPR)')
ax_base_roc.grid(True, alpha=0.3)
ax_base_roc.legend(loc='lower right', fontsize=9)

ax_manual_roc.plot(
    fpr_array_manual,
    tpr_array_manual,
    linewidth=2,
    label=f'ROC curve (AUC = {mia_auc_manual:.3f})',
)
ax_manual_roc.plot([0, 1], [0, 1], 'k--', alpha=0.6, label='Random guess')
ax_manual_roc.scatter(
    fpr_array_manual[best_threshold_index_manual],
    tpr_array_manual[best_threshold_index_manual],
    color='red',
    s=70,
    zorder=3,
    label=f'Chosen t = {t_manual:.4g}',
)
ax_manual_roc.annotate(
    f'max(TPR - FPR) = {threshold_list_manual[best_threshold_index_manual]:.3f}',
    (fpr_array_manual[best_threshold_index_manual],
     tpr_array_manual[best_threshold_index_manual]),
    xytext=(10, -28),
    textcoords='offset points',
    color='red',
)
ax_manual_roc.set_title('Manual DP-SGD')
ax_manual_roc.set_xlabel('False positive rate (FPR)')
ax_manual_roc.grid(True, alpha=0.3)
ax_manual_roc.legend(loc='lower right', fontsize=9)

for ax in (ax_base_roc, ax_manual_roc):
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)

fig.suptitle('Membership attack ROC: baseline vs manual DP-SGD', fontsize=14)
plt.tight_layout()
plt.show()

# Histogram comparison: use the same log-scale bins in both panels.
LOSS_FLOOR = 1e-8
m_plot_base_3a = np.maximum(member_signals, LOSS_FLOOR)
nm_plot_base_3a = np.maximum(nonmember_signals, LOSS_FLOOR)
m_plot_manual = np.maximum(member_signals_manual, LOSS_FLOOR)
nm_plot_manual = np.maximum(nonmember_signals_manual, LOSS_FLOOR)
hist_max_3a = max(
    m_plot_base_3a.max(),
    nm_plot_base_3a.max(),
    m_plot_manual.max(),
    nm_plot_manual.max(),
    t,
    t_manual,
)
bins_3a = np.logspace(np.log10(LOSS_FLOOR), np.log10(hist_max_3a * 1.5), 35)

fig, (ax_base_hist, ax_manual_hist) = plt.subplots(
    1, 2, figsize=(13, 4.8), sharex=True, sharey=True
)

ax_base_hist.hist(
    m_plot_base_3a,
    bins=bins_3a,
    alpha=0.55,
    label=f'Members (n={len(m_plot_base_3a)})',
    color='C0',
)
ax_base_hist.hist(
    nm_plot_base_3a,
    bins=bins_3a,
    alpha=0.55,
    label=f'Non-members (n={len(nm_plot_base_3a)})',
    color='C1',
)
ax_base_hist.axvline(t, color='k', ls='--', lw=2, label=f't = {t:.4f}')
ax_base_hist.set_xscale('log')
ax_base_hist.set_xlabel('Per-sample loss (log scale)')
ax_base_hist.set_ylabel('Count')
ax_base_hist.set_title(f'Baseline (no DP)\nAUC = {mia_auc:.3f}')
ax_base_hist.legend(fontsize=9)

ax_manual_hist.hist(
    m_plot_manual,
    bins=bins_3a,
    alpha=0.55,
    label=f'Members (n={len(m_plot_manual)})',
    color='C0',
)
ax_manual_hist.hist(
    nm_plot_manual,
    bins=bins_3a,
    alpha=0.55,
    label=f'Non-members (n={len(nm_plot_manual)})',
    color='C1',
)
ax_manual_hist.axvline(
    t_manual, color='k', ls='--', lw=2, label=f't = {t_manual:.4f}'
)
ax_manual_hist.set_xscale('log')
ax_manual_hist.set_xlabel('Per-sample loss (log scale)')
ax_manual_hist.set_title(f'Manual DP-SGD\nAUC = {mia_auc_manual:.3f}')
ax_manual_hist.legend(fontsize=9)

fig.suptitle('Loss distributions and selected threshold: baseline vs manual DP-SGD', fontsize=14)
plt.tight_layout()
plt.show()


### Phase 3a Checkpoint — Discussion

Stop here. Compare the manual DP model's **ROC, AUC, threshold-attack accuracy, and loss histogram** with Phase 2.

1. Did DP make members and non-members harder to tell apart — yes or no? Which number convinced you?
2. Would you rather release the Phase 1 model or this DP model? Why?

<details>
<summary>After you discuss — click to reveal</summary>

Yes — the loss ratio usually collapses toward ~1x, so the same attack signal is much weaker. The visible cost is often *train* accuracy (memorization), not a big drop in test accuracy. Next: Opacus runs the same clip-then-noise steps automatically.

</details>


## Phase 3b: DP-SGD with Opacus (~15 min)

**Phase 3b Goal:** Apply the DP-SGD ideas from Phase 3a using a library and observe whether membership-inference leakage decreases.

**Target:** the DP model's loss ratio moves toward 1.0, meaning members and non-members become harder to tell apart.

> We deliberately do *not* treat AUC as a pass/fail target. Phase 2 can show a moderate AUC (~0.65) together with a large loss ratio; a drop in AUC after DP is useful evidence, but a remaining moderate AUC still does not certify that the defense worked.

**What is Opacus?**

[Opacus](https://opacus.ai/) is an open-source library for training PyTorch models with differential privacy. Its `PrivacyEngine` wraps familiar PyTorch training components so we can use DP-SGD without manually implementing every privacy operation.

**What will Opacus handle for us?**

- Compute and clip per-sample gradients.
- Add calibrated Gaussian noise during optimization.
- Track the privacy budget spent during training.
- Preserve a training loop that closely resembles standard PyTorch code.

**Connection to Phase 3a:** We implemented clipping and noise ourselves to understand the mechanism. We will now use Opacus to perform those operations automatically and then compare the privacy and utility results.


15: Configure private training with Opacus

- **Goal:** Prepares a fresh model and optimizer for differentially private stochastic gradient descent.

- **What runs:** Sets the privacy hyperparameters, creates an SGD optimizer and member-data loader, and is intended to wrap them using `PrivacyEngine.make_private_with_epsilon`.

- **Outcome:** The private model, optimizer, and loader will be configured for the target epsilon and delta, and the resulting noise multiplier will be printed.

- **TODO:** Complete TODO 5 by calling `make_private_with_epsilon` with the model, optimizer, loader, epochs, privacy targets, and clipping norm.

In [ ]:
EPOCHS_DP = 20
EPSILON = 2.0
DELTA = 1e-5
MAX_GRAD_NORM = 1.0
LR_DP = 0.1

# Fresh model + plain SGD (DP-SGD builds on SGD, not Adam).
model_dp = create_model()
optimizer_dp = torch.optim.SGD(model_dp.parameters(), lr=LR_DP)

# Opacus wraps the loader to enable per-sample gradient computation.
train_loader_dp = DataLoader(member_dataset, batch_size=64, shuffle=True)


# TODO 5: Attach PrivacyEngine to make training differentially private
# Hint:
#   1. Create a PrivacyEngine to manage the private training setup and privacy accounting.
#   2. Call make_private_with_epsilon(...) and assign its three returned objects back to
#      model_dp, optimizer_dp, and train_loader_dp.
#   3. Replace each None below with the model, optimizer, loader, or privacy
#      hyperparameter variable already defined at the top of this cell.
privacy_engine = PrivacyEngine()
model_dp, optimizer_dp, train_loader_dp = privacy_engine.make_private_with_epsilon(
    module=None,          # <-- Replace None with the model
    optimizer=None,       # <-- Replace None with the optimizer
    data_loader=None,     # <-- Replace None with the training data loader
    epochs=None,          # <-- Replace None with the number of DP epochs
    target_epsilon=None,  # <-- Replace None with the privacy budget
    target_delta=None,    # <-- Replace None with delta
    max_grad_norm=None,   # <-- Replace None with the clipping bound
)

print(f"Using noise_multiplier = {optimizer_dp.noise_multiplier:.4f}")
print(f"Target: epsilon = {EPSILON}, delta = {DELTA}")

16: Train the Opacus DP model

- **Goal:** Trains the differentially private model while tracking accuracy and privacy-budget consumption.

- **What runs:** Runs forward and backward passes with `BCEWithLogitsLoss`; the Opacus-wrapped optimizer clips gradients and adds noise during `step`; `get_epsilon` tracks privacy spending.

- **Outcome:** Accuracy and epsilon are reported every five epochs, followed by the final DP test accuracy.

- **TODO:** Complete TODO 6 by implementing the private training step from `optimizer_dp.zero_grad()` through `optimizer_dp.step()`.

In [ ]:
criterion_dp = nn.BCEWithLogitsLoss()
train_accs_dp = []
test_accs_dp = []

print(f"Training DP model for {EPOCHS_DP} epochs...")
for epoch in range(EPOCHS_DP):
    model_dp.train()
    for images, labels in train_loader_dp:
        images = images.to(device)
        labels = labels.float().to(device).view(-1, 1)
        
        # TODO 6: Complete the DP training step (5 lines)
        # Hint: Clear the old gradients, run the forward pass, compute the loss,
        # backpropagate, and take an optimizer step. Opacus applies clipping and
        # noise automatically when optimizer_dp.step() is called.

        optimizer_dp.zero_grad()
        outputs = None  # <-- Replace None with forward pass
        loss = None     # <-- Replace None with loss computation
        # <-- Add backward and step calls

    if (epoch + 1) % 5 == 0:
        train_acc = evaluate_accuracy(model_dp, train_loader_dp)
        test_acc_dp_curr = evaluate_accuracy(model_dp, test_loader)
        train_accs_dp.append(train_acc)
        test_accs_dp.append(test_acc_dp_curr)
        # Track the privacy budget actually spent so far.
        eps_spent = privacy_engine.get_epsilon(DELTA)
        print(f"Epoch {epoch+1}: Train={train_acc:.1f}%, Test={test_acc_dp_curr:.1f}%, eps={eps_spent:.2f}")

test_acc_dp = evaluate_accuracy(model_dp, test_loader)
print(f"\nFinal DP model test accuracy: {test_acc_dp:.1f}%")

17: Attack and compare the DP model

- **Goal:** Repeats the membership-inference attack on the private model and compares it with the baseline.

- **What runs:** Unwraps the Opacus model when needed, obtains member and non-member signals, constructs labels and scores, explicitly loops over thresholds to compute AUC and the best `t`, plots baseline and Opacus DP-SGD ROC curves side by side, and formats a comparison table.

- **Outcome:** The ROC and table show whether differential privacy reduces the distinguishability of members and non-members.

- **TODO:** Complete TODO 7 by computing DP signals, attack labels and scores, AUC, and loss ratio using the Phase 2 procedure.

In [ ]:
model_dp_eval = model_dp._module if hasattr(model_dp, '_module') else model_dp

# TODO 7: Re-run the membership-inference attack on the DP model.
member_signals_dp = None      # <-- get signals for members
nonmember_signals_dp = None   # <-- get signals for non-members
attack_labels_dp = None       # <-- concatenate member/non-member labels
attack_scores_dp = None       # <-- concatenate negated losses
loss_ratio_dp = None          # <-- non-member/member mean-loss ratio

# Build the DP ROC curve one threshold at a time, exactly as in Phase 2.
attack_thresholds_dp = np.concatenate((
    [np.inf],
    np.sort(np.unique(attack_scores_dp))[::-1],
    [-np.inf],
))

tp_list_dp = []
fp_list_dp = []
tpr_list_dp = []
fpr_list_dp = []
num_members_dp = np.sum(attack_labels_dp == 1)
num_nonmembers_dp = np.sum(attack_labels_dp == 0)

for threshold in attack_thresholds_dp:
    predicted_member = attack_scores_dp >= threshold
    tp_dp = np.sum(predicted_member & (attack_labels_dp == 1))
    fp_dp = np.sum(predicted_member & (attack_labels_dp == 0))

    tp_list_dp.append(int(tp_dp))
    fp_list_dp.append(int(fp_dp))
    tpr_list_dp.append(tp_dp / num_members_dp)
    fpr_list_dp.append(fp_dp / num_nonmembers_dp)

tpr_array_dp = np.array(tpr_list_dp)
fpr_array_dp = np.array(fpr_list_dp)
threshold_list_dp = tpr_array_dp - fpr_array_dp
best_threshold_index_dp = int(np.argmax(threshold_list_dp))
best_score_threshold_dp = float(attack_thresholds_dp[best_threshold_index_dp])
selected_loss_threshold_dp = -best_score_threshold_dp

# Compute the DP AUC explicitly as the sum of trapezoid areas.
auc_area_dp = 0.0
for i in range(1, len(fpr_list_dp)):
    width = fpr_list_dp[i] - fpr_list_dp[i - 1]
    average_height = (tpr_list_dp[i] + tpr_list_dp[i - 1]) / 2
    auc_area_dp += width * average_height

print(f"{'Metric':<25} {'Baseline':>10} {'DP Model':>10} {'Change':>10}")
print("-" * 55)
print(f"{'MIA AUC':<25} {mia_auc:>10.4f} {auc_area_dp:>10.4f} {auc_area_dp - mia_auc:>+10.4f}")
print(f"{'Loss Ratio':<25} {loss_ratio:>10.2f}x {loss_ratio_dp:>10.2f}x {'improved' if loss_ratio_dp < loss_ratio else '':>10}")
print(f"{'Mean Loss (members)':<25} {np.mean(member_signals):>10.4f} {np.mean(member_signals_dp):>10.4f}")
print(f"{'Mean Loss (non-members)':<25} {np.mean(nonmember_signals):>10.4f} {np.mean(nonmember_signals_dp):>10.4f}")

# ROC comparison: baseline on the left, Opacus DP-SGD on the right.
fig, (ax_base_roc, ax_dp_roc) = plt.subplots(
    1, 2, figsize=(13, 5.5), sharex=True, sharey=True
)

ax_base_roc.plot(
    fpr_array,
    tpr_array,
    linewidth=2,
    label=f'ROC curve (AUC = {mia_auc:.3f})',
)
ax_base_roc.plot([0, 1], [0, 1], 'k--', alpha=0.6, label='Random guess')
ax_base_roc.scatter(
    fpr_array[best_threshold_index],
    tpr_array[best_threshold_index],
    color='red',
    s=70,
    zorder=3,
    label=f'Chosen t = {t:.4g}',
)
ax_base_roc.annotate(
    f'max(TPR - FPR) = {threshold_list[best_threshold_index]:.3f}',
    (fpr_array[best_threshold_index], tpr_array[best_threshold_index]),
    xytext=(10, -28),
    textcoords='offset points',
    color='red',
)
ax_base_roc.set_title('Baseline (no DP)')
ax_base_roc.set_xlabel('False positive rate (FPR)')
ax_base_roc.set_ylabel('True positive rate (TPR)')
ax_base_roc.grid(True, alpha=0.3)
ax_base_roc.legend(loc='lower right', fontsize=9)

ax_dp_roc.plot(
    fpr_array_dp,
    tpr_array_dp,
    linewidth=2,
    label=f'ROC curve (AUC = {auc_area_dp:.3f})',
)
ax_dp_roc.plot([0, 1], [0, 1], 'k--', alpha=0.6, label='Random guess')
ax_dp_roc.scatter(
    fpr_array_dp[best_threshold_index_dp],
    tpr_array_dp[best_threshold_index_dp],
    color='red',
    s=70,
    zorder=3,
    label=f'Chosen t = {selected_loss_threshold_dp:.4g}',
)
ax_dp_roc.annotate(
    f'max(TPR - FPR) = {threshold_list_dp[best_threshold_index_dp]:.3f}',
    (fpr_array_dp[best_threshold_index_dp], tpr_array_dp[best_threshold_index_dp]),
    xytext=(10, -28),
    textcoords='offset points',
    color='red',
)
ax_dp_roc.set_title('Opacus DP-SGD')
ax_dp_roc.set_xlabel('False positive rate (FPR)')
ax_dp_roc.grid(True, alpha=0.3)
ax_dp_roc.legend(loc='lower right', fontsize=9)

for ax in (ax_base_roc, ax_dp_roc):
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)

fig.suptitle('Membership attack ROC: baseline vs Opacus DP-SGD', fontsize=14)
plt.tight_layout()
plt.show()

# Keep the familiar name for later comparison and validation cells.
mia_auc_dp = auc_area_dp


18: Threshold attack after DP-SGD

- **Goal:** Repeats the same Youden threshold attack on the Opacus DP model and compares it to the Phase 2 baseline attack.

- **What runs:** Reuses the DP threshold selected by the explicit ROC loop, then prints accuracy and correct-count side by side with the baseline numbers from cell 12.

- **Outcome:** Draws baseline and Opacus DP-SGD loss histograms side by side, then keeps the baseline-vs-DP-vs-random accuracy comparison; the DP-classification panel remains removed. DP-SGD pushes threshold-attack accuracy toward chance (~50%), so the same attack becomes harder.

- **TODO:** N/A

Even giving the attacker the best threshold for the DP model, membership should be much harder to call than on the overfitted baseline. That is the empirical counterpart of the privacy guarantee you are buying with epsilon.


In [ ]:
# Reuse the DP operating point selected explicitly in the preceding ROC loop.
t_dp = selected_loss_threshold_dp
all_loss_dp = np.concatenate([member_signals_dp, nonmember_signals_dp])
threshold_preds_dp = (all_loss_dp <= t_dp).astype(float)
threshold_correct_dp = int((threshold_preds_dp == attack_labels_dp).sum())
threshold_acc_dp = threshold_correct_dp / len(attack_labels_dp)
tn_dp, fp_dp, fn_dp, tp_dp = confusion_matrix(
    attack_labels_dp, threshold_preds_dp
).ravel()

base_thr = {
    't': t,
    'acc': threshold_attack_acc,
    'n_correct': threshold_attack_correct,
    'n_total': len(attack_labels),
    'auc': float(mia_auc),
}
dp_thr = {
    't': t_dp,
    'acc': threshold_acc_dp,
    'n_correct': threshold_correct_dp,
    'n_total': len(attack_labels_dp),
    'tpr': float(tpr_array_dp[best_threshold_index_dp]),
    'fpr': float(fpr_array_dp[best_threshold_index_dp]),
    'auc': float(mia_auc_dp),
    'tp': int(tp_dp),
    'tn': int(tn_dp),
    'fp': int(fp_dp),
    'fn': int(fn_dp),
}

print("Threshold attack: baseline vs DP-SGD")
print("-" * 64)
print(f"{'Metric':<28} {'Baseline':>14} {'DP (eps~2)':>14}")
print("-" * 64)
print(f"{'MIA AUC':<28} {base_thr['auc']:>14.4f} {dp_thr['auc']:>14.4f}")
print(f"{'Loss threshold t':<28} {base_thr['t']:>14.6f} {dp_thr['t']:>14.6f}")
print(f"{'Attack accuracy':<28} {100*base_thr['acc']:>13.1f}% {100*dp_thr['acc']:>13.1f}%")
print(f"{'Correct / total':<28} {base_thr['n_correct']:>7d}/{base_thr['n_total']:<5d} {dp_thr['n_correct']:>7d}/{dp_thr['n_total']:<5d}")
print(f"{'vs random (50%)':<28} {'YES (better)':>14} {'closer to 50%':>14}")
print(f"  DP confusion counts: TP={tp_dp}, FN={fn_dp}, TN={tn_dp}, FP={fp_dp}")

# Phase 3b histogram comparison: baseline left, Opacus DP-SGD right.
LOSS_FLOOR = 1e-8
m_plot_base_3b = np.maximum(member_signals, LOSS_FLOOR)
nm_plot_base_3b = np.maximum(nonmember_signals, LOSS_FLOOR)
m_plot_dp = np.maximum(member_signals_dp, LOSS_FLOOR)
nm_plot_dp = np.maximum(nonmember_signals_dp, LOSS_FLOOR)
hist_max_3b = max(
    m_plot_base_3b.max(),
    nm_plot_base_3b.max(),
    m_plot_dp.max(),
    nm_plot_dp.max(),
    t,
    t_dp,
)
bins_3b = np.logspace(np.log10(LOSS_FLOOR), np.log10(hist_max_3b * 1.5), 35)

fig, (ax_base_hist, ax_dp_hist) = plt.subplots(
    1, 2, figsize=(13, 4.8), sharex=True, sharey=True
)

ax_base_hist.hist(
    m_plot_base_3b,
    bins=bins_3b,
    alpha=0.55,
    label=f'Members (n={len(m_plot_base_3b)})',
    color='C0',
)
ax_base_hist.hist(
    nm_plot_base_3b,
    bins=bins_3b,
    alpha=0.55,
    label=f'Non-members (n={len(nm_plot_base_3b)})',
    color='C1',
)
ax_base_hist.axvline(t, color='k', ls='--', lw=2, label=f't = {t:.4f}')
ax_base_hist.set_xscale('log')
ax_base_hist.set_xlabel('Per-sample loss (log scale)')
ax_base_hist.set_ylabel('Count')
ax_base_hist.set_title(f'Baseline (no DP)\nAUC = {mia_auc:.3f}')
ax_base_hist.legend(fontsize=9)

ax_dp_hist.hist(
    m_plot_dp,
    bins=bins_3b,
    alpha=0.55,
    label=f'Members (n={len(m_plot_dp)})',
    color='C0',
)
ax_dp_hist.hist(
    nm_plot_dp,
    bins=bins_3b,
    alpha=0.55,
    label=f'Non-members (n={len(nm_plot_dp)})',
    color='C1',
)
ax_dp_hist.axvline(t_dp, color='k', ls='--', lw=2, label=f't = {t_dp:.4f}')
ax_dp_hist.set_xscale('log')
ax_dp_hist.set_xlabel('Per-sample loss (log scale)')
ax_dp_hist.set_title(f'Opacus DP-SGD\nAUC = {mia_auc_dp:.3f}')
ax_dp_hist.legend(fontsize=9)

fig.suptitle('Loss distributions and selected threshold: baseline vs Opacus DP-SGD', fontsize=14)
plt.tight_layout()
plt.show()

# Phase 3b keeps this additional baseline-vs-DP-vs-random comparison.
fig, ax = plt.subplots(figsize=(7, 4))
labels_bar = ['Baseline\n(no DP)', 'DP-SGD\n(eps~2)', 'Random\nguess']
vals = [100 * base_thr['acc'], 100 * dp_thr['acc'], 50.0]
colors = ['C0', 'C2', '0.7']
bars = ax.bar(labels_bar, vals, color=colors, width=0.6)
ax.axhline(50, color='k', ls=':', lw=1)
ax.set_ylabel('Threshold-attack accuracy (%)')
ax.set_ylim(0, 100)
ax.set_title('Same attack, before vs after DP-SGD')
for bar, value in zip(bars, vals):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        value + 2,
        f'{value:.1f}%',
        ha='center',
        fontsize=11,
    )
plt.tight_layout()
plt.show()


19: Validate differential privacy results

- **Goal:** Checks whether DP reduces membership leakage while retaining useful accuracy and respecting the privacy budget.

- **What runs:** Compares baseline and DP loss ratios and accuracy gaps, evaluates DP test accuracy, queries `privacy_engine.get_epsilon`, and counts four validations.

- **Outcome:** The Phase 3 summary reports whether leakage decreased, utility remained acceptable, and epsilon stayed within budget.

- **TODO:** N/A

In [ ]:
print("=" * 50)
print("PHASE 3 VALIDATION")
print("=" * 50)
checks_passed_3 = 0
total_checks_3 = 4

if loss_ratio_dp < loss_ratio:
    print(f"[PASS] Loss ratio reduced: {loss_ratio:.2f}x -> {loss_ratio_dp:.2f}x")
    checks_passed_3 += 1
else:
    print(f"[FAIL] Loss ratio not reduced: {loss_ratio:.2f}x -> {loss_ratio_dp:.2f}x")

test_acc_dp = evaluate_accuracy(model_dp_eval, test_loader)
if test_acc_dp > 55:
    print(f"[PASS] DP model test accuracy = {test_acc_dp:.1f}% (target: > 55%)")
    checks_passed_3 += 1
else:
    print(f"[FAIL] DP model test accuracy = {test_acc_dp:.1f}% (target: > 55%)")

dp_member_acc = evaluate_accuracy(model_dp_eval, member_loader)
dp_nonmem_acc = evaluate_accuracy(model_dp_eval, nonmember_loader)
dp_acc_gap = dp_member_acc - dp_nonmem_acc
baseline_acc_gap = evaluate_accuracy(model, member_loader) - evaluate_accuracy(model, nonmember_loader)
if abs(dp_acc_gap) < abs(baseline_acc_gap):
    print(f"[PASS] Accuracy gap reduced: {baseline_acc_gap:.1f}% -> {dp_acc_gap:.1f}%")
    checks_passed_3 += 1
else:
    print(f"[FAIL] Accuracy gap not reduced: {baseline_acc_gap:.1f}% -> {dp_acc_gap:.1f}%")

eps_final = privacy_engine.get_epsilon(DELTA)
if eps_final <= EPSILON * 1.1:
    print(f"[PASS] Epsilon spent = {eps_final:.2f} (budget: {EPSILON})")
    checks_passed_3 += 1
else:
    print(f"[FAIL] Epsilon spent = {eps_final:.2f} (budget: {EPSILON})")

print("-" * 50)
print(f"Result: {checks_passed_3}/{total_checks_3} checks passed.")
if checks_passed_3 == total_checks_3:
    print("Phase 3 COMPLETE!")
print("=" * 50)

20: Compare manual DP with Opacus

- **Goal:** Compares privacy leakage and test accuracy across the baseline, Opacus DP-SGD, and manual DP-SGD models.

- **What runs:** Calls `get_membership_signal`, calculates the manual model's loss ratio with NumPy, and prints a formatted three-model comparison.

- **Outcome:** The output shows whether the hand-written clipping-and-noise procedure reproduces Opacus's privacy protection.

- **TODO:** N/A

In [ ]:
member_signals_manual = get_membership_signal(model_manual, member_loader)
nonmember_signals_manual = get_membership_signal(model_manual, nonmember_loader)
loss_ratio_manual = np.mean(nonmember_signals_manual) / (np.mean(member_signals_manual) + 1e-8)

print(f"{'Approach':<30}{'Loss Ratio':>12}{'Test Acc':>11}")
print("-" * 53)
print(f"{'Baseline (no DP)':<30}{loss_ratio:>11.2f}x{test_accs[-1]:>10.1f}%")
print(f"{'Opacus DP-SGD':<30}{loss_ratio_dp:>11.2f}x{test_acc_dp:>10.1f}%")
print(f"{'Manual DP-SGD (from scratch)':<30}{loss_ratio_manual:>11.2f}x{manual_test_acc:>10.1f}%")
print()
print("Both DP methods shrink the loss ratio toward 1.0 -- our hand-written")
print("clip + noise reproduces the privacy protection that Opacus provides.")


21: Visualize the trade-off made by the defense

- **Goal:** Puts utility and leakage side by side for the baseline, the manual DP model, and the Opacus DP model.

- **What runs:** Evaluates train and test accuracy for all three models, then draws a grouped bar chart of accuracy next to a bar chart of the membership signal.

- **Outcome:** A single figure showing how much privacy the defense buys and where the utility cost lands.

- **TODO:** N/A

The table above is easy to skim past, so this figure states the comparison explicitly. Watch **both** accuracy bars, not just the test bar: DP mostly removes the model's ability to fit its *training* data, which is exactly the memorization the attack was exploiting.

In [ ]:
names = ['Baseline\n(no DP)', 'Manual DP-SGD', 'Opacus DP-SGD']
train_acc_vals = [
    evaluate_accuracy(model, member_loader),
    evaluate_accuracy(model_manual, member_loader),
    evaluate_accuracy(model_dp_eval, member_loader),
]
test_acc_vals = [
    evaluate_accuracy(model, test_loader),
    evaluate_accuracy(model_manual, test_loader),
    evaluate_accuracy(model_dp_eval, test_loader),
]
ratio_vals = [loss_ratio, loss_ratio_manual, loss_ratio_dp]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(len(names))
width = 0.35

bars_train = ax1.bar(x - width/2, train_acc_vals, width, label='Train (members, seen)',
                     color='steelblue')
bars_test = ax1.bar(x + width/2, test_acc_vals, width, label='Test (unseen)', color='coral')
for bars in (bars_train, bars_test):
    for bar in bars:
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.0,
                 f'{bar.get_height():.0f}%', ha='center', fontsize=10)
ax1.set_xticks(x)
ax1.set_xticklabels(names)
ax1.set_ylabel('Accuracy (%)', fontsize=12)
ax1.set_ylim(0, 112)
ax1.set_title('Utility: accuracy on seen vs unseen data', fontsize=13)
ax1.legend(fontsize=10, loc='lower right')
ax1.grid(True, axis='y', alpha=0.3)

bars_ratio = ax2.bar(x, ratio_vals, 0.5, color=['firebrick', 'seagreen', 'seagreen'])
for bar in bars_ratio:
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.03,
             f'{bar.get_height():.2f}x', ha='center', fontsize=11)
ax2.axhline(y=1.0, color='gray', linestyle='--', alpha=0.8, label='No leakage (ratio = 1)')
ax2.set_xticks(x)
ax2.set_xticklabels(names)
ax2.set_ylabel('Loss ratio (non-member / member)', fontsize=12)
ax2.set_title('Privacy: DP removes the membership signal', fontsize=13)
ax2.legend(fontsize=10)
ax2.grid(True, axis='y', alpha=0.3)

plt.suptitle('DP-SGD removes the membership signal. Look at where the utility cost lands.',
             fontsize=15)
plt.tight_layout()
plt.show()

print(f"{'Model':<22}{'Train':>9}{'Test':>9}{'Loss ratio':>13}")
print("-" * 53)
for name, tr, te, ratio in zip(['Baseline (no DP)', 'Manual DP-SGD', 'Opacus DP-SGD'],
                               train_acc_vals, test_acc_vals, ratio_vals):
    print(f"{name:<22}{tr:>8.1f}%{te:>8.1f}%{ratio:>12.2f}x")

# Interpret the actual numbers rather than assuming which way they went.
d_train = train_acc_vals[2] - train_acc_vals[0]
d_test = test_acc_vals[2] - test_acc_vals[0]
print()
print(f"Opacus DP vs baseline: train {d_train:+.1f} points, test {d_test:+.1f} points, "
      f"loss ratio {ratio_vals[0]:.2f}x -> {ratio_vals[2]:.2f}x")
if d_test < -0.5:
    print("Privacy improves and test accuracy decreases: the textbook trade-off.")
else:
    print("Here DP costs almost no TEST accuracy, which may be surprising. Clipping and")
    print("noise also act as regularizers, so a model that memorizes less can generalize")
    print("just as well. The cost shows up in TRAIN accuracy, and that drop IS the")
    print("defense: the memorization it removed is precisely what the attack exploited.")
    print("On harder tasks, where fitting the training data actually requires memorizing")
    print("it, the same defense does cost real test accuracy. Phase 4 shows that regime.")


### Phase 3b Checkpoint — Discussion

Stop here. Look at your **baseline vs Opacus** numbers: loss ratio and threshold-attack accuracy.

1. Same attack as Phase 2 — did it get harder after Opacus — yes or no? Which number convinced you?
2. Manual DP-SGD vs Opacus — same defense or different? Would you ship with the library?

<details>
<summary>After you discuss — click to reveal</summary>

Yes — threshold accuracy should move toward ~50% and the loss ratio toward ~1x. Same two steps you wrote (clip, then noise); Opacus just vectorizes them and tracks ε. The library is what you would ship.

</details>

---


## Phase 4: The Privacy-Utility Trade-off (~15 min)

**Goal:** See how model utility changes as the formal privacy budget epsilon varies.

We evaluate pre-trained checkpoints so nobody has to retrain eight models live. Every checkpoint was produced with **one matched recipe** (SGD, lr = 0.1, batch 64, 60 epochs, clipping norm C = 1.0), so the only thing that changes along the sweep is epsilon. That matters: if the models also differed in optimizer or epoch budget, we could not attribute any accuracy change to privacy.

The **`fc_eps_inf.pt`** checkpoint anchors the sweep. It uses the same recipe with clipping and noise switched **off**, so it is the non-private endpoint.

The Phase 4 figure deliberately focuses on **utility**: test accuracy versus epsilon. The results table still reports loss ratio and attack AUC so you can connect the sweep to Phases 2 and 3, but those empirical measurements are not the privacy guarantee. The guarantee comes from the chosen `(epsilon, delta)`.

**TODO in this phase:** TODO 8.


22: Evaluate multiple privacy budgets

- **Goal:** Measures utility across several privacy budgets using saved model checkpoints, while retaining the familiar loss-ratio and AUC diagnostics from earlier phases.

- **What runs:** Defines `load_checkpoint` and `attack_model`, then evaluates every epsilon with the same test and membership splits.

- **Outcome:** A results table with epsilon, test accuracy, loss ratio, and attack AUC.

- **TODO:** Complete TODO 8 by computing the accuracy, membership signals, labels, and scores inside `attack_model`.


In [ ]:
import os
MODEL_TYPE = 'FC'

def load_checkpoint(name):
    '''Load a pre-trained checkpoint by name, e.g. eps_2.0 or eps_inf.'''
    fname = f"checkpoints/{MODEL_TYPE.lower()}_{name}.pt"
    model_ckpt = create_model()
    if os.path.exists(fname):
        model_ckpt.load_state_dict(torch.load(fname, map_location=device))
    else:
        # Fallback so the notebook still runs if the checkpoints are missing.
        print(f"WARNING: {fname} not found. Using a model trained in this notebook instead.")
        state = model_dp._module.state_dict() if hasattr(model_dp, '_module') else model_dp.state_dict()
        model_ckpt.load_state_dict(state)
    return model_ckpt


def attack_model(model_ckpt):
    '''Evaluate utility and the familiar Phase 2 membership diagnostics.'''
    # TODO 8: Compute the accuracy and attack inputs for this checkpoint.
    # Hint: reuse evaluate_accuracy() and get_membership_signal().
    test_acc = None   # <-- evaluate on test_loader
    signals_m = None  # <-- member losses
    signals_nm = None # <-- non-member losses
    labels = None     # <-- 1s for members, 0s for non-members
    scores = None     # <-- concatenate the negated losses

    return {
        'test_acc': test_acc,
        'loss_ratio': np.mean(signals_nm) / (np.mean(signals_m) + 1e-8),
        'mia_auc': roc_auc_score(labels, scores),
    }


# From strong privacy (0.5), through weak privacy (200), to none at all (inf).
epsilons = [0.5, 1.0, 2.0, 5.0, 10.0, 50.0, 200.0, float('inf')]
results = []

print("Evaluating models at different epsilon levels...")
print("-" * 47)
print(f"{'Epsilon':>8} {'Test Acc':>10} {'Loss Ratio':>12} {'MIA AUC':>10}")
print("-" * 47)
for eps in epsilons:
    name = 'eps_inf' if eps == float('inf') else f'eps_{eps}'
    metrics = attack_model(load_checkpoint(name))
    results.append({'epsilon': eps, **metrics})
    eps_str = "inf" if eps == float('inf') else f"{eps:g}"
    print(f"{eps_str:>8} {metrics['test_acc']:>9.1f}% "
          f"{metrics['loss_ratio']:>11.2f}x {metrics['mia_auc']:>10.4f}")


23: Plot the privacy-utility trade-off

- **Goal:** Visualizes how utility changes as the formal privacy budget varies.

- **What runs:** Extracts test accuracy from `results` and draws one utility curve across the epsilon sweep. The non-private endpoint uses a star so it is not mistaken for a DP setting.

- **Outcome:** A single plot of test accuracy versus epsilon. Smaller epsilon means stronger privacy; the figure shows the utility cost of tightening that budget.

- **TODO:** N/A


In [ ]:
eps_labels = ['inf' if r['epsilon'] == float('inf') else f"{r['epsilon']:g}"
              for r in results]
accs = [r['test_acc'] for r in results]

# The eps=inf point has no clipping and no noise, so it is not a DP model at all.
# Draw it with a different marker so nobody reads it as part of the DP curve.
dp_idx = [i for i, r in enumerate(results) if r['epsilon'] != float('inf')]
inf_idx = [i for i, r in enumerate(results) if r['epsilon'] == float('inf')]

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.plot(dp_idx, [accs[i] for i in dp_idx], 'bo-', markersize=8, label='DP-SGD')
ax.plot(inf_idx, [accs[i] for i in inf_idx], 'k*', markersize=18,
        label='No DP (no clipping, no noise)')
ax.set_xticks(range(len(results)))
ax.set_xticklabels(eps_labels)
ax.set_xlabel('Epsilon (privacy budget)', fontsize=12)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Utility vs Privacy Budget', fontsize=14)
ax.legend(fontsize=10, loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


24: Validate the privacy-budget sweep

- **Goal:** Checks that every privacy setting was evaluated and that loosening the DP budget produces a visible utility gain.

- **What runs:** Verifies the result count, then compares accuracy at the strongest privacy setting with the best finite-epsilon DP accuracy.

- **Outcome:** The Phase 4 summary reports whether the complete sweep ran and whether the expected privacy-utility pattern is visible.

- **TODO:** N/A


In [ ]:
print("=" * 50)
print("PHASE 4 VALIDATION")
print("=" * 50)
checks_passed_4 = 0
total_checks_4 = 2

if len(results) == len(epsilons):
    print(f"[PASS] Evaluated {len(results)} privacy settings")
    checks_passed_4 += 1
else:
    print(f"[FAIL] Expected {len(epsilons)} results, got {len(results)}")

# Utility should improve as the budget loosens, across the range where noise still binds.
# We compare within the DP models only: eps=inf has no clipping and is a separate endpoint.
finite = [r for r in results if r['epsilon'] != float('inf')]
acc_strong = finite[0]['test_acc']
acc_best_dp = max(r['test_acc'] for r in finite)
if acc_best_dp > acc_strong + 1.0:
    print(f"[PASS] Loosening the budget buys utility: eps={finite[0]['epsilon']:g} gives "
          f"{acc_strong:.1f}%, the best DP setting gives {acc_best_dp:.1f}%")
    checks_passed_4 += 1
else:
    print(f"[WARN] Expected a visible accuracy gain from eps={finite[0]['epsilon']:g} "
          f"({acc_strong:.1f}%) to the best DP setting ({acc_best_dp:.1f}%)")

print("-" * 50)
print(f"Result: {checks_passed_4}/{total_checks_4} checks passed.")
if checks_passed_4 == total_checks_4:
    print("Phase 4 COMPLETE!")
print("=" * 50)


### Phase 4 Checkpoint — Final Discussion

Stop here. Look at the **single utility plot**: test accuracy versus epsilon.

1. Does a smaller epsilon usually cost accuracy — yes or no?
2. Can this utility curve alone tell you whether a model is private — yes or no?

<details>
<summary>After you discuss — click to reveal</summary>

1. **Yes.** Smaller epsilon means stronger noise and a tighter privacy guarantee, and the strongest settings have the lowest test accuracy here.

2. **No.** Accuracy measures utility, not privacy. The formal privacy statement comes from the `(epsilon, delta)` guarantee; empirical attacks such as the ones in Phases 2 and 3 can demonstrate leakage, but failing to find leakage does not certify privacy.

</details>

---

### Where this goes next

In this lab we worked from the defender's side: we saw what membership signals look like and how DP can suppress them. In the follow-up challenge you will switch to the attacker's side and try to exploit those signals in a larger model.
